In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import confusion_matrix, classification_report
import joblib

In [2]:
df = pd.read_csv('../data/final_features_logs.csv')

df = df.sort_values('timestamp')

X = df.drop(['timestamp', 'machine_id', 'failure'], axis=1)
y = df['failure']

split_idx = int(len(df) * 0.8)

X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("--- Data Split and Scaling Completed ---")
print(f"Training Features Shape: {X_train_scaled.shape}")

--- Data Split and Scaling Completed ---
Training Features Shape: (2396, 5)


In [3]:
log_model=LogisticRegression(class_weight='balanced',random_state=42)
log_model.fit(X_train_scaled, y_train)

y_pred_log=log_model.predict(X_test_scaled)

print("Confusion matrix:\n", confusion_matrix(y_test,y_pred_log))
print("\nClassification report:\n", classification_report(y_test,y_pred_log))

Confusion matrix:
 [[549  27]
 [  1  23]]

Classification report:
               precision    recall  f1-score   support

           0       1.00      0.95      0.98       576
           1       0.46      0.96      0.62        24

    accuracy                           0.95       600
   macro avg       0.73      0.96      0.80       600
weighted avg       0.98      0.95      0.96       600



In [4]:
rf_model=RandomForestClassifier(random_state=42, class_weight='balanced')
rf_model.fit(X_train_scaled, y_train)

y_pred_rf=rf_model.predict(X_test_scaled)

print("Random Forest Confusion Matrix:\n",confusion_matrix(y_test,y_pred_rf))

print("\nRandom Forest Classification report:\n", classification_report(y_test,y_pred_rf))

Random Forest Confusion Matrix:
 [[575   1]
 [  2  22]]

Random Forest Classification report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       576
           1       0.96      0.92      0.94        24

    accuracy                           0.99       600
   macro avg       0.98      0.96      0.97       600
weighted avg       0.99      0.99      0.99       600



In [5]:
xgboost_model=XGBClassifier(random_state=42, max_depth=3, learning_rate=0.1)
xgboost_model.fit(X_train_scaled,y_train)

y_pred_xgboost=xgboost_model.predict(X_test_scaled)

print("Confusion Matrix:\n", confusion_matrix(y_test,y_pred_xgboost))
print("\nClassification report:\n", classification_report(y_test,y_pred_xgboost))

Confusion Matrix:
 [[575   1]
 [  2  22]]

Classification report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       576
           1       0.96      0.92      0.94        24

    accuracy                           0.99       600
   macro avg       0.98      0.96      0.97       600
weighted avg       0.99      0.99      0.99       600



In [6]:
# Saving the champion model and the scaler permanently for deployment
joblib.dump(rf_model,'../models/random_forest_model.pkl')
joblib.dump(scaler,'../models/robust_scaler.pkl')


print("---Champion Model and Scaler saved successfully---")


---Champion Model and Scaler saved successfully---
